# Gemma-E4B: Climate pilot

Same protocols as the main study (baseline single-image like/scroll + logprobs, single-image across the 6
`metrics/realistic` engagement scales + logprobs, full 7x7 paired A/B `metrics` grid), pointed
at `climate_pilot/posts/`.

**25 posts (not 50/100): this is a scoped pilot, not a full replication.**

In [1]:
import sys, subprocess

# 1. Uninstall torchaudio
subprocess.run([sys.executable, "-m", "pip", "uninstall", "torchaudio", "-y"])

# 2. Install PyTorch with CUDA 12.4
subprocess.run([sys.executable, "-m", "pip", "install",
    "torch==2.6.0", "torchvision==0.21.0",
    "--index-url", "https://download.pytorch.org/whl/cu124",
    "--user", "-q"], check=True)

# 3. Install latest transformers and accelerate (allowing pip to pull compatible tokenizers naturally)
subprocess.run([sys.executable, "-m", "pip", "install",
    "git+https://github.com/huggingface/transformers",
    "accelerate",
    "--user", "-q"], check=True)

print("✅ Installation complete — restart the kernel now")



✅ Installation complete — restart the kernel now


Restart kernel after running the setup cell above.

In [2]:
!nvidia-smi

Tue Aug 25 13:24:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.127.08             Driver Version: 550.127.08     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:CF:00.0 Off |                   On |
| N/A   26C    P0             76W /  700W |                  N/A   |     N/A      Default |
|                                         |                        |              Enabled |
+-----------------------------------------+-----

In [3]:
# --- HF Auth ---
import sys
sys.path.append("/home/jovyan")
from config_hf_token import HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN)

# --- Path setup ---
from pathlib import Path
ROOT_DIR = Path().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR / "experiments/e1"))

# --- Load Gemma model ---
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "google/gemma-4-E4B-it"
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, padding_side="left")
device = model.device

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

In [4]:
from e1_utils.sampling import build_paired_sample
from e1_utils.e1_optimized import (
    LIKE_PROMPT_SINGLE, LIKE_PROMPT_YESNO, LIKE_PROMPT_PAIR, ADJACENT_PAIRS,
    run_e1_baseline, run_e1_metrics, run_e1_metrics_paired
)
from e1_utils.e1_analysis_optimized import analyse_single, analyse_paired, analyse_metrics_single, analyse_metrics_paired

# --- Configuration: climate pilot, NOT the main benchmarking/ pool ---
EXPERIMENT_DIR = Path().resolve().parent        # experiments/e1_climate/  -- shared across all 4 climate-pilot models, so they see the identical 25-image sample
OUTPUT_DIR = Path().resolve() / "outputs"
SEED = 42
SAMPLE_SIZE = 25

correct_dir = ROOT_DIR / "climate_pilot/posts/correct/PNGs"
incorrect_dir = ROOT_DIR / "climate_pilot/posts/incorrect/PNGs"
all_images = build_paired_sample(correct_dir, incorrect_dir, SEED, SAMPLE_SIZE, EXPERIMENT_DIR)
selected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]

# metrics/realistic condition only -- the condition that showed the strongest conformity effect
# in the main study (Section 6.2)
correct_base = ROOT_DIR / "climate_pilot/posts/correct/PNGs/metrics/realistic"
incorrect_base = ROOT_DIR / "climate_pilot/posts/incorrect/PNGs/metrics/realistic"


📋 Loading existing selection from /home/jovyan/conformity-llms-facebook-posts/experiments/e1_climate/selected_images.json
✅ All selected numbers verified in both correct and incorrect folders.
Selected 25 pairs → 50 images total


In [5]:
import torch
free, total = torch.cuda.mem_get_info(0)
print(torch.cuda.get_device_name(0))
print(f"VRAM total   : {total / 1e9:.1f} GB")
print(f"VRAM free    : {free / 1e9:.1f} GB   (device-wide, all processes)")
print(f"this process : {torch.cuda.memory_reserved(0) / 1e9:.1f} GB reserved")

NVIDIA H100 80GB HBM3 MIG 3g.40gb
VRAM total   : 42.3 GB
VRAM free    : 26.0 GB   (device-wide, all processes)
this process : 15.9 GB reserved


In [6]:
from e1_utils.inference_gemma import run_inference_gemma

In [7]:
from e1_utils.e1_optimized import (
    run_e1_baseline_logprobs, run_e1_metrics_logprobs,
    LIKE_CANDIDATES_SINGLE, LIKE_CANDIDATES_YESNO
)

from e1_utils.inference_gemma import run_inference_with_scores_gemma

## Approach 1 -- single image, like/scroll, baseline (0 engagement)

In [8]:
run_e1_baseline(all_images, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_baseline.json",
              inference_fn=run_inference_gemma)

✅ 001_correct → like
✅ 001_incorrect → like
✅ 002_correct → like
✅ 002_incorrect → like
✅ 003_correct → like
✅ 003_incorrect → like
✅ 004_correct → like
✅ 004_incorrect → like
✅ 005_correct → like
✅ 005_incorrect → like
✅ 006_correct → like
✅ 006_incorrect → like
✅ 007_correct → like
✅ 007_incorrect → like
✅ 008_correct → like
✅ 008_incorrect → like
✅ 009_correct → like
✅ 009_incorrect → like
✅ 010_correct → like
✅ 010_incorrect → like
✅ 011_correct → like
✅ 011_incorrect → like
✅ 012_correct → like
✅ 012_incorrect → like
✅ 013_correct → like
✅ 013_incorrect → like
✅ 014_correct → like
✅ 014_incorrect → like
✅ 015_correct → like
✅ 015_incorrect → like
✅ 016_correct → like
✅ 016_incorrect → like
✅ 017_correct → like
✅ 017_incorrect → like
✅ 018_correct → like
✅ 018_incorrect → like
✅ 019_correct → like
✅ 019_incorrect → like
✅ 020_correct → like
✅ 020_incorrect → like
✅ 021_correct → like
✅ 021_incorrect → like
✅ 022_correct → like
✅ 022_incorrect → like
✅ 023_correct → like
✅ 023_incor

In [9]:
run_e1_baseline_logprobs(all_images, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, candidates=LIKE_CANDIDATES_SINGLE,
              output_filename="e1_results_baseline_logprobs.json", score_fn=run_inference_with_scores_gemma)

✅ 001_correct → like {'like': {'logprob': -19.143293380737305, 'prob_forced_choice': 0.9999996423647644}, 'scroll': {'logprob': -33.98704528808594, 'prob_forced_choice': 3.5763523558576065e-07}}
✅ 001_incorrect → like {'like': {'logprob': -21.168275833129883, 'prob_forced_choice': 0.9999989322991665}, 'scroll': {'logprob': -34.918277740478516, 'prob_forced_choice': 1.067700833566654e-06}}
✅ 002_correct → like {'like': {'logprob': -23.339323043823242, 'prob_forced_choice': 0.999997097682479}, 'scroll': {'logprob': -36.08932113647461, 'prob_forced_choice': 2.9023175209211066e-06}}
✅ 002_incorrect → like {'like': {'logprob': -24.51803970336914, 'prob_forced_choice': 0.9999948260812298}, 'scroll': {'logprob': -36.68991470336914, 'prob_forced_choice': 5.173918770225734e-06}}
✅ 003_correct → like {'like': {'logprob': -22.074724197387695, 'prob_forced_choice': 0.9999995111687903}, 'scroll': {'logprob': -36.60597229003906, 'prob_forced_choice': 4.888312097062957e-07}}
✅ 003_incorrect → like {'

In [10]:
analyse_single(OUTPUT_DIR, "e1_results_baseline.json", like_answer="like")


Single image analysis: e1_results_baseline
=== Summary ===


,metric,value
0,overall_like_rate_%,100.0
1,like_rate_correct_%,100.0
2,like_rate_incorrect_%,100.0


=== Per Image Results ===


,image,variant,prompt,answer
0,001_correct,correct,You are shown a social media post.\nYou can ei...,like
1,001_incorrect,incorrect,You are shown a social media post.\nYou can ei...,like
2,002_correct,correct,You are shown a social media post.\nYou can ei...,like
3,002_incorrect,incorrect,You are shown a social media post.\nYou can ei...,like
4,003_correct,correct,You are shown a social media post.\nYou can ei...,like
5,003_incorrect,incorrect,You are shown a social media post.\nYou can ei...,like
6,004_correct,correct,You are shown a social media post.\nYou can ei...,like
7,004_incorrect,incorrect,You are shown a social media post.\nYou can ei...,like
8,005_correct,correct,You are shown a social media post.\nYou can ei...,like
9,005_incorrect,incorrect,You are shown a social media post.\nYou can ei...,like


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/experiments/e1_climate/gemma4-e4b/outputs/e1_analysis_baseline.csv


## Approach 1 variant -- single image, like/scroll, across the 6 `metrics/realistic` engagement scales

In [11]:
run_e1_metrics(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_metrics.json",
              inference_fn=run_inference_gemma)

✅ 001_correct_10 → like
✅ 001_incorrect_10 → like
✅ 002_correct_10 → like
✅ 002_incorrect_10 → like
✅ 003_correct_10 → like
✅ 003_incorrect_10 → like
✅ 004_correct_10 → like
✅ 004_incorrect_10 → like
✅ 005_correct_10 → like
✅ 005_incorrect_10 → like
✅ 006_correct_10 → like
✅ 006_incorrect_10 → like
✅ 007_correct_10 → like
✅ 007_incorrect_10 → like
✅ 008_correct_10 → like
✅ 008_incorrect_10 → like
✅ 009_correct_10 → like
✅ 009_incorrect_10 → like
✅ 010_correct_10 → like
✅ 010_incorrect_10 → like
✅ 011_correct_10 → like
✅ 011_incorrect_10 → like
✅ 012_correct_10 → like
✅ 012_incorrect_10 → like
✅ 013_correct_10 → like
✅ 013_incorrect_10 → like
✅ 014_correct_10 → like
✅ 014_incorrect_10 → like
✅ 015_correct_10 → like
✅ 015_incorrect_10 → like
✅ 016_correct_10 → like
✅ 016_incorrect_10 → like
✅ 017_correct_10 → like
✅ 017_incorrect_10 → like
✅ 018_correct_10 → like
✅ 018_incorrect_10 → like
✅ 019_correct_10 → like
✅ 019_incorrect_10 → like
✅ 020_correct_10 → like
✅ 020_incorrect_10 → like


In [12]:
run_e1_metrics_logprobs(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, candidates=LIKE_CANDIDATES_SINGLE,
              output_filename="e1_results_metrics_logprobs.json", score_fn=run_inference_with_scores_gemma)

✅ 001_correct_10 → like {'like': {'logprob': -19.1392765045166, 'prob_forced_choice': 0.9999996640327807}, 'scroll': {'logprob': -34.045528411865234, 'prob_forced_choice': 3.3596721931138406e-07}}
✅ 001_incorrect_10 → like {'like': {'logprob': -20.654966354370117, 'prob_forced_choice': 0.9999991684703864}, 'scroll': {'logprob': -34.654964447021484, 'prob_forced_choice': 8.315296136781726e-07}}
✅ 002_correct_10 → like {'like': {'logprob': -22.82025718688965, 'prob_forced_choice': 0.9999956431283266}, 'scroll': {'logprob': -35.16400909423828, 'prob_forced_choice': 4.356871673422428e-06}}
✅ 002_incorrect_10 → like {'like': {'logprob': -23.651737213134766, 'prob_forced_choice': 0.999994744604293}, 'scroll': {'logprob': -35.807987213134766, 'prob_forced_choice': 5.255395707074638e-06}}
✅ 003_correct_10 → like {'like': {'logprob': -21.52656364440918, 'prob_forced_choice': 0.9999991684735583}, 'scroll': {'logprob': -35.52656555175781, 'prob_forced_choice': 8.315264416531168e-07}}
✅ 003_incorr

In [13]:
analyse_metrics_single(OUTPUT_DIR, "e1_results_metrics.json", like_answer="like")


Metrics single image analysis: e1_results_metrics
=== Overall Summary ===


,metric,value
0,overall_like_rate_%,100.0
1,like_rate_correct_%,100.0
2,like_rate_incorrect_%,100.0


=== Rate per Scale Value ===


/home/jovyan/conformity-llms-facebook-posts/experiments/e1/e1_utils/e1_analysis_optimized.py:116: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  per_scale = df.groupby("scale_value").apply(lambda g: pd.Series({


,scale_value,total_images,overall_like_rate_%,like_rate_correct_%,like_rate_incorrect_%
0,10,50.0,100.0,100.0,100.0
1,100,50.0,100.0,100.0,100.0
2,1000,50.0,100.0,100.0,100.0
3,10000,50.0,100.0,100.0,100.0
4,100000,50.0,100.0,100.0,100.0
5,1000000,50.0,100.0,100.0,100.0


=== Per Image Results ===


,image,num,variant,scale_value,prompt,answer
0,001_correct_10,001,correct,10,You are shown a social media post.\nYou can ei...,like
2,002_correct_10,002,correct,10,You are shown a social media post.\nYou can ei...,like
4,003_correct_10,003,correct,10,You are shown a social media post.\nYou can ei...,like
6,004_correct_10,004,correct,10,You are shown a social media post.\nYou can ei...,like
8,005_correct_10,005,correct,10,You are shown a social media post.\nYou can ei...,like
...,...,...,...,...,...,...
291,021_incorrect_1000000,021,incorrect,1000000,You are shown a social media post.\nYou can ei...,like
293,022_incorrect_1000000,022,incorrect,1000000,You are shown a social media post.\nYou can ei...,like
295,023_incorrect_1000000,023,incorrect,1000000,You are shown a social media post.\nYou can ei...,like
297,024_incorrect_1000000,024,incorrect,1000000,You are shown a social media post.\nYou can ei...,like


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/experiments/e1_climate/gemma4-e4b/outputs/e1_analysis_metrics.csv


## Approach 2 -- paired A/B forced choice, full 7x7 `metrics/realistic` disparity grid

In [ ]:
import time
start = time.time()
run_e1_metrics_paired(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR, SEED,
              prompt=LIKE_PROMPT_PAIR, output_filename="e1_results_metrics_paired.json",
              inference_fn=run_inference_gemma)
elapsed = time.time() - start
print(f"\n⏱ Total runtime: {elapsed/60:.1f} min")

✅ 001_correct0_vs_incorrect0 → liked correct (answered A)
✅ 002_correct0_vs_incorrect0 → liked correct (answered A)
✅ 003_correct0_vs_incorrect0 → liked correct (answered A)
✅ 004_correct0_vs_incorrect0 → liked incorrect (answered A)
✅ 005_correct0_vs_incorrect0 → liked correct (answered A)
✅ 006_correct0_vs_incorrect0 → liked incorrect (answered A)
✅ 007_correct0_vs_incorrect0 → liked correct (answered A)
✅ 008_correct0_vs_incorrect0 → liked correct (answered A)
✅ 009_correct0_vs_incorrect0 → liked correct (answered A)
✅ 010_correct0_vs_incorrect0 → liked incorrect (answered A)
✅ 011_correct0_vs_incorrect0 → liked incorrect (answered A)
✅ 012_correct0_vs_incorrect0 → liked incorrect (answered A)
✅ 013_correct0_vs_incorrect0 → liked correct (answered A)
✅ 014_correct0_vs_incorrect0 → liked incorrect (answered A)
✅ 015_correct0_vs_incorrect0 → liked correct (answered A)
✅ 016_correct0_vs_incorrect0 → liked incorrect (answered A)
✅ 017_correct0_vs_incorrect0 → liked correct (answered A)


In [ ]:
analyse_metrics_paired(OUTPUT_DIR, "e1_results_metrics_paired.json")